In [2]:
%pip install pyttsx3

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install pywin32

'%pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import cv2
import dlib
import pyttsx3
import numpy as np
import time
import winsound
import subprocess
from scipy.spatial import distance

def init_engine():
    try:
        # Explicitly use SAPI5 on Windows for better compatibility
        return pyttsx3.init(driverName='sapi5')
    except Exception:
        return None

engine = init_engine()
if engine is not None:
    try:
        engine.setProperty('rate', 165)
        engine.setProperty('volume', 1.0)
    except Exception:
        pass

cam = cv2.VideoCapture(0)

face_detector = dlib.get_frontal_face_detector()

# loading the location of the predictor file
predictor = dlib.shape_predictor('shape_predictor_68_face_landmarks.dat')

# function to calculate the eye aspect ratio
def detecting_eye(eye):
    point_a = distance.euclidean(eye[1], eye[5])
    point_b = distance.euclidean(eye[2], eye[4])
    point_c = distance.euclidean(eye[0], eye[3])
    return (point_a + point_b) / (2.0 * point_c)

def draw_eye(frame, eye_points, color):
    eye_array = np.array(eye_points, dtype=np.int32)
    cv2.polylines(frame, [eye_array], True, color, 2, cv2.LINE_AA)
    for point in eye_points:
        cv2.circle(frame, point, 2, color, -1, cv2.LINE_AA)

last_alert_time = 0.0
alert_cooldown_seconds = 2.0
drowsy_threshold = 0.27
low_eye_frames = 0
low_eye_frames_required = 4

def speak_with_powershell(text):
    # Backup speech path using Windows SAPI via PowerShell
    command = (
        "Add-Type -AssemblyName System.Speech; "
        "$speak = New-Object System.Speech.Synthesis.SpeechSynthesizer; "
        f"$speak.Speak('{text}')"
    )
    try:
        subprocess.Popen([
            'powershell',
            '-NoProfile',
            '-ExecutionPolicy',
            'Bypass',
            '-Command',
            command,
        ])
        return True
    except Exception:
        return False

def play_alert():
    global last_alert_time
    now = time.time()
    if now - last_alert_time < alert_cooldown_seconds:
        return

    last_alert_time = now
    text = 'Alert wake up'

    if engine is not None:
        try:
            engine.stop()
            engine.say(text)
            engine.runAndWait()
            return
        except Exception:
            pass

    if speak_with_powershell(text):
        return

    # Last fallback when speech paths fail
    winsound.Beep(2200, 700)

while True:
    null, frame = cam.read()

    if not null or frame is None:
        continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector(gray)

    for face in faces:
        face_landmarks = predictor(gray, face)
        left_eye = []
        right_eye = []

        for n in range(36, 42):
            x = face_landmarks.part(n).x
            y = face_landmarks.part(n).y
            left_eye.append((x, y))

        for n in range(42, 48):
            x = face_landmarks.part(n).x
            y = face_landmarks.part(n).y
            right_eye.append((x, y))

        draw_eye(frame, left_eye, (0, 255, 0))
        draw_eye(frame, right_eye, (0, 255, 255))

        right_eye_aspect_ratio = detecting_eye(right_eye)
        left_eye_aspect_ratio = detecting_eye(left_eye)
        eye_rat = round((right_eye_aspect_ratio + left_eye_aspect_ratio) / 2, 2)

        left = face.left()
        top = face.top()
        right = face.right()
        bottom = face.bottom()
        cv2.rectangle(frame, (left, top), (right, bottom), (255, 0, 0), 2)
        cv2.putText(frame, f'EAR: {eye_rat:.2f}', (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        if eye_rat < drowsy_threshold:
            low_eye_frames += 1
        else:
            low_eye_frames = 0

        cv2.putText(
            frame,
            f'Low-EAR frames: {low_eye_frames}/{low_eye_frames_required}',
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 0),
            2,
        )

        if low_eye_frames >= low_eye_frames_required:
            cv2.putText(frame, 'Drowsiness Detected', (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
            cv2.putText(frame, 'WAKE UP', (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
            # Keep alerting while drowsiness persists; cooldown controls repeat rate
            play_alert()

    cv2.imshow('Drowsiness Detection window', frame)
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()